# Model training

## The shape contract

`sequence_to_tensor` returns **one sample. There is no batch axis in it.**

| call | returns | reading | pairs with |
| --- | --- | --- | --- |
| `sequence_to_tensor(series_dir)` | `(24, 224, 224, 1)` | `(K, H, W, C)` - the 24 slices are a *depth* axis, 1 channel because the images are monochrome | `Conv3D` |
| `sequence_to_tensor(series_dir, as_channels=True)` | `(224, 224, 24)` | `(H, W, K)` - the 24 slices are *channels* | `Conv2D` |

The batch axis is added by **stacking samples**, so a batch of `B` series is
`(B, 24, 224, 224, 1)` or `(B, 224, 224, 24)`. A batch of one is `x[None, ...]`,
i.e. `(1, 24, 224, 224, 1)` - that is where the leading `1` comes from, and it
belongs to the data, never to `layers.Input(shape=...)`.

`layers.Input(shape=...)` takes the **per-sample** shape; Keras adds the batch
axis itself. So `Input(shape=(1, 24, 224, 224, 1))` declares a rank-6 tensor
`(None, 1, 24, 224, 224, 1)`, while `Conv3D` needs rank 5 `(B, d, h, w, c)`.
That was the bug. Below, the shape is never hardcoded at all - it is read off
the data with `x.shape[1:]`, which drops the batch axis and leaves exactly the
per-sample shape.

Labels are separate: the twelve `soft_*` columns of `data/meta/train_index.csv`,
one row per series, so `(B, 12)` against the model's `(B, 12)` output.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import callbacks as keras_callbacks
from tensorflow.keras import layers, models

# functions/ is two levels up from notebooks/david/
sys.path.append(str(Path.cwd().parents[1] / "functions"))
from sequence_to_tensor import K, IMG_SIZE, pick_series, sequence_to_tensor  # noqa: E402

DATA = Path("../../data")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# --- point these at the real data when it lands -------------------------------
TRAIN_IMAGES = DATA / "train_series"                     # <study>/<series>/*.dcm
GOLD_IMAGES = DATA / "savetheknees_gold_images"
DERIVED_LABELS = DATA / "meta" / "derived_labels.csv"    # StudyInstanceUID + the 12 columns

CACHE = DATA / "tensor_cache"                            # one .npy per study, written once
# ------------------------------------------------------------------------------

# The training images and derived labels are not in this checkout yet. Step 5 skips
# itself while that is true, so the notebook still runs top to bottom; the plumbing
# test at the end exercises the same functions on the gold images that ARE here.
HAVE_TRAINING_DATA = TRAIN_IMAGES.exists() and DERIVED_LABELS.exists()
print("training data present:", HAVE_TRAINING_DATA)

2026-08-28 11:17:32.214229: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-28 11:17:32.225162: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-28 11:17:32.335011: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-28 11:17:32.469480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-28 11:17:32.615194: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

2026-08-28 11:17:35.089602: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


training data present: False


## Building the model

The layout decides the convolution, so the builder reads the rank of one sample
and picks:

* rank 4 `(K, H, W, 1)` -> a **3D CNN**, the slices are a depth axis the kernels
  move through, so the model can see across slices.
* rank 3 `(H, W, K)` -> a **2.5D CNN**, the slices are channels, so every kernel
  already sees all 24 at once but the model has no notion of slice order.

Two things that were in the first draft are gone:

* **`TimeDistributed` is dropped.** It exists to apply a layer at every step of a
  *time* axis, and there is no time axis here. Wrapped around the pooling it made
  Keras read the 24 slices as time and hand a rank-4 slice to a 3D pool, and
  wrapped around the last `Dense` it produced `(B, T, 12)` against labels of
  `(B, 12)`.
* **the duplicated `Dense(20)`.** Both lines read `shared_feature`, so the second
  overwrote the first and one layer was silently dropped. They are chained now.

In [2]:
def build_model_3d(input_shape, n_labels):
    """rank 4 (K, H, W, 1) -> a 3D CNN.

    Conv3D kernels move through depth as well as height and width, so a filter
    sees the same spot on three neighbouring slices at once and the model can
    learn how a finding develops across the stack.
    """
    inputs = layers.Input(shape=input_shape)

    # Block 1. (1, 2, 2) pools the two image axes but keeps all 24 slices: there
    # are only 24 of them against 224 pixels, so depth is the axis we can least
    # afford to spend first.        (24, 224, 224, 1) -> (24, 112, 112, 32)
    net = layers.Conv3D(32, 3, padding="same", activation="relu")(inputs)
    net = layers.MaxPooling3D(pool_size=(1, 2, 2), padding="same")(net)

    # Block 2. Now all three axes.  (24, 112, 112, 32) -> (12, 56, 56, 64)
    net = layers.Conv3D(64, 3, padding="same", activation="relu")(net)
    net = layers.MaxPooling3D(pool_size=(2, 2, 2), padding="same")(net)

    # Block 3. All three again.     (12, 56, 56, 64) -> (6, 28, 28, 64)
    net = layers.Conv3D(64, 3, padding="same", activation="relu")(net)
    net = layers.MaxPooling3D(pool_size=(2, 2, 2), padding="same")(net)

    # Global pooling, not Flatten: it collapses every spatial axis to one number
    # per filter, so the head does not depend on the image size.
    #                               (6, 28, 28, 64) -> (64,)
    net = layers.GlobalAveragePooling3D()(net)

    return models.Model(inputs=inputs, outputs=classification_head(net, n_labels))


def build_model_25d(input_shape, n_labels):
    """rank 3 (H, W, K) -> a 2.5D CNN.

    The 24 slices are channels, so every Conv2D kernel already reads all 24 at
    once - but channels are unordered, so unlike the 3D model this one cannot
    tell slice 5 from slice 20. It is far cheaper to train, which is the trade.
    """
    inputs = layers.Input(shape=input_shape)

    # Block 1.                      (224, 224, 24) -> (112, 112, 32)
    net = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    net = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(net)

    # Block 2.                      (112, 112, 32) -> (56, 56, 64)
    net = layers.Conv2D(64, 3, padding="same", activation="relu")(net)
    net = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(net)

    # Block 3.                      (56, 56, 64) -> (28, 28, 64)
    net = layers.Conv2D(64, 3, padding="same", activation="relu")(net)
    net = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(net)

    # Same reason as in the 3D model.
    #                               (28, 28, 64) -> (64,)
    net = layers.GlobalAveragePooling2D()(net)

    return models.Model(inputs=inputs, outputs=classification_head(net, n_labels))

def classification_head(features, n_labels):
    """The tail both models share: (64,) of pooled features -> one score per label.

    The two Dense(20) layers are chained, not both read off `features` - that was
    the bug in the first draft, where the second silently replaced the first.

    sigmoid, not softmax: softmax would make the 12 scores sum to 1, i.e. "pick
    exactly one finding". A knee can carry several at once, so each label gets an
    independent 0-1 score.
    """
    net = layers.Dense(20, activation="relu")(features)
    net = layers.Dense(20, activation="relu")(net)
    return layers.Dense(n_labels, activation="sigmoid")(net)


def build_model(input_shape, n_labels=len(LABELS)):
    """One sample's shape -> a compile-ready model.

    input_shape is the shape of ONE series, with no batch axis:
        (24, 224, 224, 1) -> 3D CNN     (sequence_to_tensor default)
        (224, 224, 24)    -> 2.5D CNN   (as_channels=True)
    """
    if len(input_shape) == 4:
        return build_model_3d(input_shape, n_labels)
    if len(input_shape) == 3:
        return build_model_25d(input_shape, n_labels)
    raise ValueError(
        f"one sample is (K, H, W, 1) or (H, W, K), got {input_shape}. "
        "If this has a leading 1, you passed a batch of one instead of a sample."
    )


def compile_model(model):
    """Attach the optimizer, loss, and metrics. Separate from build_model so the
    architecture and the training setup can be changed independently, and so a
    model can be rebuilt without re-deciding how it is trained.

    binary_crossentropy, not categorical: it scores each of the 12 sigmoid outputs
    on its own, which is what "several findings at once" needs. It also takes soft
    targets (floats in [0, 1]) unchanged, which is what the derived labels are.
    """
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
        ],
    )
    return model

## Step 1 - the labels

Two label sources, doing two different jobs:

| | studies | where the labels come from | job |
| --- | --- | --- | --- |
| **derived** | ~4350 | the Claude API reads each free-text `Report`, translates it, and scores the twelve findings | **train** |
| **gold** | 58 | a human read them; the rows of `data/train.csv` that are not `NaN` | **validate** |

Training on derived and validating on gold is deliberately this way round. The
derived labels will have their own quirks - the extractor may systematically miss a
finding, or over-call another. If validation used derived labels too, the model
could reproduce those quirks perfectly and score wonderfully, and you would never
find out. Scoring against gold asks the question you actually care about: does the
model agree with a radiologist?

For that to mean anything the 58 gold studies must be **held out of training
completely**, even where a derived label also exists for them. A model tested on
something it trained on tells you what it memorised, not what it learned.
`derived_labels(exclude=...)` is what enforces the separation.

### What is not in this checkout yet

Only `data/savetheknees_gold_images/` is here - the 58 gold studies. The ~4350
training images and the derived label table have not landed. The three paths at the
top of the next cell are the only lines to change when they do.

In [3]:
# --- point these at the real data when it lands -------------------------------
TRAIN_IMAGES = DATA / "train_series"                     # <study>/<series>/*.dcm
GOLD_IMAGES = DATA / "savetheknees_gold_images"
DERIVED_LABELS = DATA / "meta" / "derived_labels.csv"    # StudyInstanceUID + the 12 columns

CACHE = DATA / "tensor_cache"                            # one .npy per study, written once
# ------------------------------------------------------------------------------


def gold_labels(train_csv=DATA / "train.csv"):
    """The 58 human-read studies: the rows with all twelve labels filled in.

    This is the validation set, and the only labels here a radiologist wrote.
    """
    train = pd.read_csv(train_csv)
    gold = train[train[LABELS].notna().all(axis=1)]
    return gold[["StudyInstanceUID"] + LABELS].reset_index(drop=True)


def derived_labels(path=DERIVED_LABELS, exclude=()):
    """The Claude-derived labels: the training set.

    `exclude` drops study UIDs that must not be trained on. Pass the gold UIDs, so
    a study present in both tables cannot leak into training and quietly inflate
    the validation score.

    Values may be soft (floats in [0, 1]) rather than 0/1. They are NOT rounded:
    binary_crossentropy accepts soft targets, and a 0.6 from the extractor carries
    real information that thresholding to 1 would throw away.
    """
    table = pd.read_csv(path)

    missing = [c for c in LABELS if c not in table.columns]
    if missing:
        raise ValueError(f"derived label table is missing columns: {missing}")

    keep = table[~table.StudyInstanceUID.isin(set(exclude))]
    keep = keep[keep[LABELS].notna().all(axis=1)]
    return keep[["StudyInstanceUID"] + LABELS].reset_index(drop=True)


def load_study(study_uid, series_df, images_root, axis="X", as_channels=False):
    """One StudyInstanceUID -> one tensor, or None if there is no usable series.

    `sequence_to_tensor` takes a SERIES directory and a study holds several, so
    `pick_series` chooses one (fluid-sensitive first). The module's own
    study_to_tensor hardcodes data_root/train_series/, which is not where the gold
    images live, so the two are composed directly instead.
    """
    series_uid = pick_series(study_uid, series_df, axis)
    if series_uid is None:
        return None

    series_dir = Path(images_root) / study_uid / series_uid
    if not series_dir.is_dir():
        return None

    return sequence_to_tensor(series_dir, as_channels=as_channels)

## Step 2 - cache the images

Reading one study means opening ~24 DICOM files, decoding the pixels, resizing and
normalising them. That takes about 0.2 s, so the ~4350 training studies take roughly
15 minutes.

An **epoch** is one pass over the training data, and training takes many epochs. If
the decoding happened inside the training loop, those 15 minutes would be paid again
every epoch, every time redoing byte-for-byte identical work. So it is done once,
here, and saved as one small `.npy` file per study. Training then reads those.

The cache stores **uint8** (whole numbers 0-255) rather than float32 (decimals).
`sequence_to_tensor` has already normalised every image to the range [0, 1], so
storing 0-255 loses about 1/255 of the precision and takes a quarter of the disk:
roughly 5 GB instead of 21 GB. `load_cached` divides by 255 to get the decimals back.

In [4]:
def cache_path(study_uid, cache_dir=CACHE, axis="X"):
    return Path(cache_dir) / f"{study_uid}_{axis}.npy"


def build_cache(index, series_df, images_root, cache_dir=CACHE, axis="X", verbose=True):
    """Decode every study once, store it as uint8, return the UIDs that worked.

    Safe to re-run: a study already on disk is skipped, so an interrupted run
    resumes rather than starting over. Studies with no readable series are simply
    absent from the returned list, which is what keeps labels aligned later.
    """
    Path(cache_dir).mkdir(parents=True, exist_ok=True)

    ok = []
    for n, study_uid in enumerate(index.StudyInstanceUID, start=1):
        out = cache_path(study_uid, cache_dir, axis)

        if not out.exists():
            x = load_study(study_uid, series_df, images_root, axis)   # float32 in [0, 1]
            if x is None:
                continue
            np.save(out, (x * 255).round().astype("uint8"))

        ok.append(study_uid)
        if verbose and n % 250 == 0:
            print(f"cached {n}/{len(index)} studies")

    return ok


def cached_subset(index, uids):
    """Cut the label table down to the studies that actually made it into the cache.

    Why this is needed: build_cache skips studies it cannot read, so afterwards the
    label table has rows the cache has no image for. If the two ever drift apart,
    study A gets trained against study B's labels and nothing raises an error - the
    model just quietly learns nonsense.

    The merge rebuilds the table in cache order, so row i of the labels and study i
    of the cache are the same knee, by construction.
    """
    order = pd.DataFrame({"StudyInstanceUID": list(uids)})
    return order.merge(index, on="StudyInstanceUID", how="left").reset_index(drop=True)


def load_cached(study_uid, cache_dir=CACHE, axis="X", as_channels=False):
    """One cached study back to the exact contract sequence_to_tensor produces:
    float32 in [0, 1], (24, 224, 224, 1) or (224, 224, 24).
    """
    x = np.load(cache_path(study_uid, cache_dir, axis)).astype("float32") / 255.0
    return np.ascontiguousarray(np.transpose(x[..., 0], (1, 2, 0))) if as_channels else x

## Step 3 - feed the images to the model

The whole training set as float32 is ~21 GB, which will not fit in memory. So the
data is **streamed**: the model asks for the next batch, and only those few studies
are read off disk. Memory holds one batch at a time instead of the corpus.

`tf.data.Dataset` is TensorFlow's tool for this. Read the chain at the bottom of
`make_dataset` in order, each step feeding the next:

* `from_tensor_slices` - start from the list of study IDs and their labels
* `shuffle` - put them in a random order, different every epoch, so the model never
  sees the same sequence twice and cannot learn the order itself
* `map(read)` - load the image for each ID
* `batch(8)` - group them into batches of 8, because the model trains on batches
* `prefetch` - prepare the next batch while the model is busy with the current one

Note the order: shuffling happens **before** loading, on a list of ID strings. That
is nearly free. Shuffling after loading would need a buffer big enough to hold the
tensors, which is the 21 GB this design exists to avoid.

In [5]:
def make_dataset(uids, labels, batch_size=8, shuffle=False, cache_dir=CACHE,
                 axis="X", as_channels=False):
    """(study UIDs, label array) -> a batched tf.data.Dataset reading the cache.

    uids and labels must be row-aligned: labels[i] belongs to uids[i].
    """
    uids = np.asarray(uids)
    labels = np.asarray(labels, dtype="float32")
    if len(uids) != len(labels):
        raise ValueError(f"{len(uids)} studies but {len(labels)} label rows")

    sample_shape = (IMG_SIZE, IMG_SIZE, K) if as_channels else (K, IMG_SIZE, IMG_SIZE, 1)

    # This runs once per study, when the dataset is asked for the next batch.
    # The decorator only silences a warning: TensorFlow likes to rewrite functions
    # into graph code, cannot read the source of a function defined in a notebook,
    # and says so loudly. Nothing here needs rewriting.
    @tf.autograph.experimental.do_not_convert
    def read(uid, y):
        # np.load is ordinary Python, and a tf.data pipeline is not Python - it is
        # a graph. numpy_function is the door between the two: it lets a plain
        # Python function run inside the pipeline.
        #
        # What comes back through that door has no shape attached, so we say what
        # it is. Without set_shape, Keras cannot work out the model input shape
        # and the error appears much later, somewhere less obvious.
        x = tf.numpy_function(
            lambda u: load_cached(u.decode(), cache_dir, axis, as_channels),
            [uid], tf.float32)
        x.set_shape(sample_shape)
        y.set_shape((len(LABELS),))
        return x, y

    ds = tf.data.Dataset.from_tensor_slices((uids, labels))
    if shuffle:
        ds = ds.shuffle(len(uids), reshuffle_each_iteration=True)   # UIDs, not tensors

    return (ds.map(read, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(batch_size)
              .prefetch(tf.data.AUTOTUNE))

## Step 4 - train

Three separate functions, one job each:

* `build_model` - what the network looks like
* `compile_model` - how it learns (optimizer, loss, metrics)
* `train_model` - actually fit it to the data

Keeping them apart is what lets you train more epochs, resume later, or train a
model you loaded from a file. A single function that builds, compiles and fits in
one call cannot do any of that - calling it twice throws the first model away and
starts from scratch.

**The callbacks** run at the end of each epoch and are what stop this from being a
guessing game:

* `EarlyStopping` - stop when the gold AUC has not improved for 5 epochs, and put
  back the best weights seen, not whatever the last epoch happened to produce
* `ModelCheckpoint` - save the best model to a file as it goes
* `ReduceLROnPlateau` - when progress stalls, halve the learning rate and take
  smaller steps

All three watch `val_auc` - the AUC on the gold set. `mode="max"` says higher is
better; the default guesses from the metric's name and guesses wrong for a custom
one, so it is always worth stating.

**AUC**, roughly: pick a knee that has the finding and one that does not. AUC is the
chance the model gives the first a higher score than the second. 0.5 is coin-flip,
1.0 is perfect. It is used here instead of accuracy because the findings are rare -
a model that answers "no" to everything scores high on accuracy and is useless.

In [6]:
def training_callbacks(checkpoint_path="best_model.keras", patience=5):
    """Stop when gold AUC stops improving, and keep the best weights.

    Without these the epoch count is a guess, and the model you end up with is
    whatever the last epoch produced rather than the best one seen.
    """
    return [
        keras_callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=patience, restore_best_weights=True),
        keras_callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_auc", mode="max", save_best_only=True),
        keras_callbacks.ReduceLROnPlateau(
            monitor="val_auc", mode="max", factor=0.5, patience=max(1, patience // 2)),
    ]


def train_model(model, train_ds, val_ds=None, epochs=10, callbacks=None):
    """Fit an already-compiled model. Returns the Keras History.

    Callbacks are only attached when there is a validation set for them to watch.
    """
    if val_ds is None:
        return model.fit(train_ds, epochs=epochs)

    return model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=training_callbacks() if callbacks is None else callbacks,
    )


def evaluate_per_label(model, ds, labels):
    """AUC per finding on the gold set, not one number for all twelve.

    A single averaged AUC hides everything: gold prevalence runs from 35 positives
    (Effusion) to 9 (MCL), so one strong common label carries the average while the
    rare ones sit at chance. With 58 studies these numbers are noisy either way -
    `positives` is printed alongside so a score can be read with its sample size.

    A label with only one class present has no defined AUC and comes back NaN.
    """
    probs = model.predict(ds, verbose=0)
    truth = np.asarray(labels, dtype="float32")

    rows = []
    for i, name in enumerate(LABELS):
        y, score = truth[:, i], probs[:, i]
        if len(np.unique(y)) < 2:
            auc = np.nan
        else:
            metric = tf.keras.metrics.AUC()
            metric.update_state(y, score)
            auc = float(metric.result())
        rows.append({"label": name, "positives": int(y.sum()), "auc": auc})

    return pd.DataFrame(rows).set_index("label")

## Step 5 - the real run

Everything above is definitions; this is the only part that touches data.

These cells skip themselves until the training images and the derived label table
are in place - that is what `HAVE_TRAINING_DATA` guards. Once the data lands they
run as written, no edits needed.

The caching cell is the slow one, roughly 15 minutes for ~4350 studies, once. After
it nothing opens a DICOM again and the epochs are fast.

In [7]:
series_df = pd.read_csv(DATA / "train_series.csv")
gold = gold_labels()

print(f"{len(gold)} gold studies (validation)")
print(gold[LABELS].sum().astype(int).to_string())

58 gold studies (validation)
ACL                 24
MCL                  9
Medial Meniscus     26
Lateral Meniscus    23
Medial OA           15
Lateral OA          11
PF OA               21
Effusion            35
Synovitis           27
Baker's             12
Contusion           19
Fracture            18


In [8]:
if HAVE_TRAINING_DATA:
    train_index = derived_labels(exclude=gold.StudyInstanceUID)
    print(f"{len(train_index)} derived studies to train on")
else:
    print("skipped: no derived label table yet")

skipped: no derived label table yet


In [9]:
if HAVE_TRAINING_DATA:
    # Decode once. Re-running is cheap: anything already cached is skipped.
    gold = cached_subset(gold, build_cache(gold, series_df, GOLD_IMAGES))
    train_index = cached_subset(train_index, build_cache(train_index, series_df, TRAIN_IMAGES))

    print(f"{len(train_index)} train and {len(gold)} gold studies cached")
else:
    print("skipped: no training images yet")

skipped: no training images yet


In [10]:
if HAVE_TRAINING_DATA:
    train_ds = make_dataset(train_index.StudyInstanceUID, train_index[LABELS],
                            batch_size=8, shuffle=True)
    gold_ds = make_dataset(gold.StudyInstanceUID, gold[LABELS], batch_size=8)

    # The per-sample shape comes off the data, never hardcoded.
    one_sample = next(iter(train_ds))[0].shape[1:]
    print("one sample", one_sample)
else:
    print("skipped: nothing to stream yet")

skipped: nothing to stream yet


In [11]:
if HAVE_TRAINING_DATA:
    model = compile_model(build_model(one_sample))
    model.summary()
else:
    print("skipped: no data to read the input shape from")

skipped: no data to read the input shape from


In [12]:
if HAVE_TRAINING_DATA:
    history = train_model(model, train_ds, gold_ds, epochs=10)
else:
    print("skipped: nothing to train on")

skipped: nothing to train on


In [13]:
if HAVE_TRAINING_DATA:
    display(evaluate_per_label(model, gold_ds, gold[LABELS]))
else:
    print("skipped: no trained model to score")

skipped: no trained model to score


## Plumbing test on the 58 gold studies

The training images are not here yet, so Step 5 above skips itself. This section
runs the *same functions* on the only images that are in this checkout - the 58 gold
studies - to prove the machinery works end to end: cache, stream, build, compile,
train, evaluate.

**This is a plumbing test, not an experiment.** The gold studies are the validation
set, and training on them is exactly what the pipeline is built to prevent. It
happens here only because they are the images available, and the model is thrown
away afterwards. Nothing below feeds into Step 5.

To keep it structurally honest the 58 are split 40/18, so the reported AUCs are at
least computed on studies the model did not see. With 18 studies those numbers are
still far too noisy to mean anything - the rarest findings will have one or two
positives. **What is being checked is that the code runs and the shapes line up, not
whether the model is any good.**

In [14]:
# Decode the gold studies into the cache. ~15 s the first time, instant after that.
gold = cached_subset(gold, build_cache(gold, series_df, GOLD_IMAGES))
print(f"{len(gold)} gold studies cached")

58 gold studies cached


In [15]:
# Split the 58 into 40 fit / 18 score, so the AUCs are at least out-of-sample.
# Shuffled first: the table is in UID order, which has nothing to do with the
# findings, but an unshuffled split is a habit worth not forming.
shuffled = gold.sample(frac=1, random_state=0).reset_index(drop=True)
fit_part, score_part = shuffled.iloc[:40], shuffled.iloc[40:]

fit_ds = make_dataset(fit_part.StudyInstanceUID, fit_part[LABELS],
                      batch_size=8, shuffle=True)
score_ds = make_dataset(score_part.StudyInstanceUID, score_part[LABELS], batch_size=8)

print(f"{len(fit_part)} to fit, {len(score_part)} to score")
print("one sample", next(iter(fit_ds))[0].shape[1:])

40 to fit, 18 to score
one sample (24, 224, 224, 1)


In [16]:
# The 3D model: the same three calls Step 5 makes.
# A separate checkpoint name, so a throwaway model cannot overwrite a real one.
test_model = compile_model(build_model(next(iter(fit_ds))[0].shape[1:]))
test_history = train_model(test_model, fit_ds, score_ds, epochs=3,
                           callbacks=training_callbacks("plumbing_test_3d.keras"))

Epoch 1/3


1/5 ━━━━━━━━━━━━━━━━━━━━ 25s 6s/step - accuracy: 0.1250 - auc: 0.4722 - loss: 0.6940 - precision: 0.3019 - recall: 0.5517

2/5 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.1250 - auc: 0.4618 - loss: 0.6947 - precision: 0.3156 - recall: 0.4817 

3/5 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.1250 - auc: 0.4614 - loss: 0.6947 - precision: 0.3342 - recall: 0.4683

4/5 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.1250 - auc: 0.4640 - loss: 0.6946 - precision: 0.3509 - recall: 0.4632

...

2/5 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.3750 - auc: 0.5475 - loss: 0.6795 - precision: 0.3438 - recall: 0.3414 

3/5 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.3889 - auc: 0.5507 - loss: 0.6789 - precision: 0.3588 - recall: 0.3367

4/5 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.3854 - auc: 0.5528 - loss: 0.6772 - precision: 0.3656 - recall: 0.3334

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3783 - auc: 0.5470 - loss: 0.6781 - precision: 0.3707 - recall: 0.3274

5/5 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step - accuracy: 0.3500 - auc: 0.5236 - loss: 0.6817 - precision: 0.3913 - recall: 0.3034 - val_accuracy: 0.5556 - val_auc: 0.5872 - val_loss: 0.6434 - val_precision: 0.3704 - val_recall: 0.3226 - learning_rate: 0.0010


In [17]:
evaluate_per_label(test_model, score_ds, score_part[LABELS])

,positives,auc
label,,
ACL,10,0.525000
MCL,2,0.437500
Medial Meniscus,5,0.469231
Lateral Meniscus,6,0.625000
Medial OA,1,0.823529
Lateral OA,4,0.732143
PF OA,5,0.546154
Effusion,11,0.818182
Synovitis,6,0.312500


In [18]:
# And the 2.5D layout: as_channels=True is the only change. build_model reads the
# rank of one sample and returns a Conv2D stack instead of a Conv3D one.
fit_ds_25d = make_dataset(fit_part.StudyInstanceUID, fit_part[LABELS],
                          batch_size=8, shuffle=True, as_channels=True)
score_ds_25d = make_dataset(score_part.StudyInstanceUID, score_part[LABELS],
                            batch_size=8, as_channels=True)

model_25d = compile_model(build_model(next(iter(fit_ds_25d))[0].shape[1:]))
train_model(model_25d, fit_ds_25d, score_ds_25d, epochs=3,
            callbacks=training_callbacks("plumbing_test_25d.keras"))

Epoch 1/3


1/5 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.2500 - auc: 0.5426 - loss: 0.6928 - precision: 0.3600 - recall: 0.4865

2/5 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.1875 - auc: 0.5141 - loss: 0.6937 - precision: 0.3535 - recall: 0.4730

3/5 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.1528 - auc: 0.5130 - loss: 0.6936 - precision: 0.3601 - recall: 0.4789

4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.1380 - auc: 0.5169 - loss: 0.6934 - precision: 0.3621 - recall: 0.4851

...

2/5 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.0938 - auc: 0.5882 - loss: 0.6856 - precision: 0.5203 - recall: 0.5586 

3/5 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.0764 - auc: 0.5985 - loss: 0.6830 - precision: 0.5026 - recall: 0.5756

4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.0651 - auc: 0.6004 - loss: 0.6816 - precision: 0.4931 - recall: 0.5853

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.0571 - auc: 0.6028 - loss: 0.6806 - precision: 0.4905 - recall: 0.5907

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - accuracy: 0.0250 - auc: 0.6123 - loss: 0.6763 - precision: 0.4802 - recall: 0.6124 - val_accuracy: 0.0000e+00 - val_auc: 0.5402 - val_loss: 0.6773 - val_precision: 0.3506 - val_recall: 0.4355 - learning_rate: 0.0010


In [19]:
evaluate_per_label(model_25d, score_ds_25d, score_part[LABELS])

,positives,auc
label,,
ACL,10,0.643750
MCL,2,0.437500
Medial Meniscus,5,0.500000
Lateral Meniscus,6,0.548611
Medial OA,1,0.794118
Lateral OA,4,0.785714
PF OA,5,0.638462
Effusion,11,0.168831
Synovitis,6,0.708333
